<a href="https://colab.research.google.com/github/PatilVaishnav131/disease-prediction-ai/blob/main/dengue_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Dengue** virus detection using Raman Spectrascopy images and Deep Transfer Learning. It contains dataset, matlab code and readme file for further use.

In [18]:
# Step 1: Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Import libraries
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import shutil
import pathlib


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Step 3: Paths
original_dir = "/content/drive/MyDrive/DENGUE_DATA_DNN"
base_dir = "/content/DENGUE_SPLIT"  # working copy inside Colab

# Create fresh directory
if os.path.exists(base_dir):
    shutil.rmtree(base_dir)
os.makedirs(base_dir)

# Split train/val/test = 70/15/15
for category in ["Healthy", "Diseased"]:
    files = os.listdir(os.path.join(original_dir, category))
    np.random.shuffle(files)
    total = len(files)
    train_split = int(0.7 * total)
    val_split = int(0.85 * total)

    splits = {
        "train": files[:train_split],
        "val": files[train_split:val_split],
        "test": files[val_split:]
    }

    for split, split_files in splits.items():
        split_path = os.path.join(base_dir, split, category)
        os.makedirs(split_path, exist_ok=True)
        for f in split_files:
            shutil.copy(os.path.join(original_dir, category, f), os.path.join(split_path, f))


In [ ]:
# Step 4: Check distribution
for split in ["train", "val", "test"]:
    healthy = len(os.listdir(os.path.join(base_dir, split, "Healthy")))
    diseased = len(os.listdir(os.path.join(base_dir, split, "Diseased")))
    print(f"{split}: Healthy={healthy}, Diseased={diseased}")

In [ ]:
# Step 5: Plot dataset distribution
counts = {
    "Train Healthy": len(os.listdir(os.path.join(base_dir, "train", "Healthy"))),
    "Train Diseased": len(os.listdir(os.path.join(base_dir, "train", "Diseased"))),
    "Val Healthy": len(os.listdir(os.path.join(base_dir, "val", "Healthy"))),
    "Val Diseased": len(os.listdir(os.path.join(base_dir, "val", "Diseased"))),
    "Test Healthy": len(os.listdir(os.path.join(base_dir, "test", "Healthy"))),
    "Test Diseased": len(os.listdir(os.path.join(base_dir, "test", "Diseased"))),
}
plt.bar(counts.keys(), counts.values(), color=['green','red','green','red','green','red'])
plt.xticks(rotation=45)
plt.ylabel("Number of Images")
plt.title("Dataset Distribution")
plt.show()


In [ ]:
# Step 6: Image Generators
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=10,
                                   zoom_range=0.1,
                                   horizontal_flip=True)

test_val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    os.path.join(base_dir, "train"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

val_gen = test_val_datagen.flow_from_directory(
    os.path.join(base_dir, "val"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

test_gen = test_val_datagen.flow_from_directory(
    os.path.join(base_dir, "test"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)


In [ ]:
# Step 7: Visualize some sample images
x_batch, y_batch = next(train_gen)
plt.figure(figsize=(10,10))
for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(x_batch[i])
    plt.title("Healthy" if y_batch[i]==0 else "Diseased")
    plt.axis("off")
plt.suptitle("Sample Training Images")
plt.show()

In [ ]:
# Step 8: Build CNN
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation="relu", input_shape=(128,128,3)),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(64, (3,3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(128, (3,3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])
model.summary()

In [ ]:
# Step 9: Train
history = model.fit(train_gen,
                    validation_data=val_gen,
                    epochs=15)


In [ ]:

# Step 10: Plot training curves
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label="Train Acc")
plt.plot(history.history['val_accuracy'], label="Val Acc")
plt.legend(); plt.title("Accuracy")

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label="Train Loss")
plt.plot(history.history['val_loss'], label="Val Loss")
plt.legend(); plt.title("Loss")
plt.show()


In [ ]:

# Step 11: Evaluate on Test Data
test_gen.reset()
y_true = test_gen.classes
y_pred = (model.predict(test_gen) > 0.5).astype("int32")

print("Classification Report:\n", classification_report(y_true, y_pred, target_names=["Healthy","Diseased"]))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Healthy","Diseased"], yticklabels=["Healthy","Diseased"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()
